# Treinamento R(2+1)D + LSTM Multimodal — vídeo + áudio (sem Optuna)

Versão sem busca de hiperparâmetros: treina diretamente com os valores definidos na seção de configuração. Treinamento usando a **(2+1)D CNN** (`r2plus1d_18`, pré-treinada na Kinetics-400) como extrator de vídeo. O áudio continua sendo processado pelo AudioMAE (congelado).

### Diferença importante vs. a versão framewise (ResNet2D)
A (2+1)D CNN processa o clipe inteiro com convoluções espaço-temporais, então
o eixo `T` é reduzido pelos strides temporais da rede (aprox. `T' = T // 8`
depois do stem + layer1..4). A sequência que chega na LSTM aqui é mais curta
do que na versão framewise — isso é esperado, já que a CNN 3D já captura
parte da dinâmica temporal sozinha; a LSTM modela uma temporalidade "mais
grossa" por cima.


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import os, sys, datetime, random, gc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

from transformers import AutoModel
import einops
import timm
import torchaudio

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

from models.nn.r2plus1d_lstm_multimodal import R2Plus1DLSTM_MultiModal
from preprocessing.loader_multimodal_frac2 import (
    build_multimodal_dataloader,
    train_video_transform,
    TARGET_SHAPE,
    train_mel_transform,
)
from preprocessing.balanced_dataset import balanced_df


/mnt/storage_C4/gaussian_football/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    # Habilita TF32 para ops de matriz — sem perda de qualidade notável, +15 % velocidade
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # autotuner de kernels

SEED = 435
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Torch: 2.5.1+cu121
Torchvision: 0.20.1+cu121
CUDA disponível: True
CUDA: 12.1
GPU: NVIDIA RTX A4500
Device: cuda


## 2. Configuração

In [4]:
LABELS_ALL  = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_combinadas_wg.csv"
LABELS_DIR  = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used"
CKPT_DIR    = "/mnt/storage_C4/gaussian_football/models/checkpoints"
RUNS_DIR    = "/mnt/storage_C4/gaussian_football/runs_multimodal_r2plus1d"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# Treino 
EPOCHS       = 100
PATIENCE     = 20
LR           = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
MONITOR      = "loss"   # "ccc" | "mae" | "loss"

# Loss / treino (antes escolhidos pelo Optuna — agora fixos)
FREEZE_MODE     = "finetune"   # "frozen" | "finetune" | "full"
UNFREEZE_EPOCH  = 5
USE_FUSION      = True
ALWAYS_BN       = False
ALPHA           = 1.0          # peso do termo de ranking na CombinedLoss
MARGIN_SCALE    = 1.0
LR_BACKBONE     = None         # None = mesmo lr pra tudo; ou defina um valor menor p/ a CNN

# DataLoader
BATCH_SIZE   = 2   # a (2+1)D CNN consome bem mais VRAM que a ResNet2D por causa
                    # das convoluções 3D — comece pequeno e ajuste GRAD_ACCUM/FRAME_STEP

# GPU / memória 
USE_AMP        = True   # Mixed Precision FP16 — principal economia de VRAM
USE_COMPILE    = True   # torch.compile (PyTorch >= 2.0); desative se der erro
GRAD_ACCUM     = 4      # acumula N mini-batches antes de step → simula batch*N
GRAD_CKPT      = True   # gradient checkpointing real nos blocos da (2+1)D CNN
FRAME_STEP     = 1      # subamostragem temporal ANTES da CNN 3D (1 = usa todos os frames)
FRAC_EPOCH     = 0.1    # usa 10 % dos grupos por época
NUM_WORKERS    = 2      # menos workers = menos vídeos simultâneos na RAM

# Modelo
MODEL_DEFAULTS = dict(
    frame_step=FRAME_STEP,
    hidden_size=256,
    num_layers=1,
    use_dropout=True,
    dropout_p=0.3,
    in_channels=1,           # grayscale, igual ao pipeline de vídeo atual
    use_grad_checkpoint=GRAD_CKPT,
)


## 3. Balanceando os Dados

In [5]:
FORCE_REBUILD = True

paths_balanced = {
    "train": os.path.join(LABELS_DIR, "balanced_todas_as_ligas_train_wg.csv"),
    "valid": os.path.join(LABELS_DIR, "balanced_todas_as_ligas_valid_wg.csv"),
    "test":  os.path.join(LABELS_DIR, "balanced_todas_as_ligas_test_wg.csv"),
}

if FORCE_REBUILD or not all(os.path.exists(p) for p in paths_balanced.values()):
    all_data = pd.read_csv(LABELS_ALL)
    for split, out_path in paths_balanced.items():
        subset  = all_data[all_data["split"] == split]
        balanced = balanced_df(subset, "game_id", threshold=0.5, random_state=SEED)
        balanced.to_csv(out_path, index=False)
        print(f"{split}: {len(balanced)} clipes -> {out_path}")
else:
    print("CSVs balanceados já existem.")


train: 16912 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_train_wg.csv
valid: 6152 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_valid_wg.csv
test: 5730 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_test_wg.csv


## 4. DataLoaders Multimodais

In [6]:
TRAIN_PATH = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_train_wg.csv"
VAL_PATH   = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_valid_wg.csv"
TEST_PATH  = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_test_wg.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)


In [7]:
# Filtra apenas goal / Background
eventos_usados = ["goal", "Background"]
dir_bg = "/mnt/storage_C4/gaussian_football/data/datasets_goal_background"
os.makedirs(dir_bg, exist_ok=True)

for nome, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    filt = df[df["type"].isin(eventos_usados)]
    filt.to_csv(f"{dir_bg}/{nome}.csv", index=False)
    print(f"{nome}: {filt['type'].value_counts().to_dict()}")

TRAIN_PATH = f"{dir_bg}/train.csv"
VAL_PATH   = f"{dir_bg}/val.csv"
TEST_PATH  = f"{dir_bg}/test.csv"


train: {'Background': 56396, 'goal': 8460}
val: {'Background': 17917, 'goal': 3076}
test: {'Background': 18697, 'goal': 2865}


In [8]:
GROUPS            = True
GROUPS_COLUMN_ID  = "window_id"

def _make_loader(csv_path, split, pair, batch_size, shuffle):
    """Fábrica de DataLoader com configurações GPU-otimizadas."""
    return build_multimodal_dataloader(
        csv_path=csv_path,
        pair=pair,
        split=split,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        is_grayscale=True,
        pin_memory=False, # True esgota RAM fixada com vídeos grandes
        groups=GROUPS,
        column_groups_id=GROUPS_COLUMN_ID,
        video_transform=train_video_transform if shuffle else None,
        mel_transform=train_mel_transform if shuffle else None,
        target_shape=TARGET_SHAPE,
        epoch_frac=FRAC_EPOCH,
        frame_step=FRAME_STEP,
    )

train_loader = _make_loader(TRAIN_PATH, "train", pair=True,  batch_size=BATCH_SIZE, shuffle=True)
val_loader   = _make_loader(VAL_PATH,   "valid", pair=True,  batch_size=BATCH_SIZE, shuffle=False)


Dataset de Treino: 62066/64856 exemplos válidos.
6144 grupos encontrados.
Low: 53977
High: 8089

897 pares de grupos criados.

Dataset de Validação: 18642/20993 exemplos válidos.
1827 grupos encontrados.
Low: 15961
High: 2681

295 pares de grupos criados.



## 5. Métricas

In [9]:
def calculate_ccc(y_true, y_pred):
    """Concordance Correlation Coefficient sobre dois arrays 1-D."""
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true,  var_pred  = np.var(y_true),  np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    denom = var_true + var_pred + (mean_true - mean_pred) ** 2
    return 0.0 if denom == 0 else (2 * cov) / denom

def binary_accuracy(y_true, y_pred, threshold=0.5):
    return float(np.mean((y_pred > threshold) == (y_true > threshold)))


## 6. Loss Combinada (BCE + Margin Ranking Adaptativa)

In [10]:
class CombinedLoss(nn.Module):
    """BCE + Margin Ranking Loss com margem adaptativa."""

    def __init__(self, alpha=1.0, margin_scale=1.0):
        super().__init__()
        self.alpha = alpha
        self.margin_scale = margin_scale
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, low_label, high_label, low_pred, high_pred, return_components=False):
        bce = (self.bce(low_pred, low_label) + self.bce(high_pred, high_label)) / 2
        margin = self.margin_scale * (high_label - low_label)
        rank = torch.relu(margin - (high_pred - low_pred)).mean()
        loss = bce + self.alpha * rank
        return (loss, bce, rank) if return_components else loss


## 7. Treino GPU-Otimizado

In [11]:
MONITOR_MODE = {"ccc": "max", "mae": "min", "loss": "min"}


def _apply_freeze(model, freeze_backbone):
    """Liga/desliga gradiente da (2+1)D CNN; o stem (índice 0 de model.cnn,
    que inclui o primeiro conv adaptado para grayscale) sempre treina."""
    for name, param in model.cnn.named_parameters():
        param.requires_grad = name.startswith("0.") or (not freeze_backbone)


def _set_backbone_eval(model):
    """Congela BatchNorm na (2+1)D CNN, mantém o stem (índice 0) em train."""
    for i, module in enumerate(model.cnn):
        (module.train if i == 0 else module.eval)()


def _is_better(current, best, mode):
    return current > best if mode == "max" else current < best


def _log_pred_scatter(writer, y_true, y_pred, tag, step):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(y_true, y_pred, s=8, alpha=0.4)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "--", color="gray", linewidth=1)
    ax.set_xlabel("target"); ax.set_ylabel("pred"); ax.set_title("pred vs target")
    writer.add_figure(tag, fig, step)
    plt.close(fig)


def train_model(
    model, train_loader, val_loader, *,
    criterion, run_name, optimizer, scheduler,
    loss_name,
    freeze_mode="finetune", unfreeze_epoch=5,
    freeze_bn_always=True,
    epochs=EPOCHS, patience=PATIENCE, monitor=MONITOR,
    grad_clip=GRAD_CLIP,
    use_amp=USE_AMP,
    accum_steps=GRAD_ACCUM,
    checkpoint_path=None, device=device,
    lr=LR,
):
    """
    Treina e valida por época com Mixed Precision + acumulação de gradiente.
    """
    model.to(device)
    gc.collect()
    torch.cuda.empty_cache()

    if checkpoint_path is None:
        checkpoint_path = os.path.join(CKPT_DIR, f"{run_name}.pth")
    last_path = os.path.join(CKPT_DIR, f"{run_name}_last.pth")

    mode = MONITOR_MODE[monitor]
    best_score = -np.inf if mode == "max" else np.inf
    best_metrics, best_true, best_pred = {}, None, None
    epochs_no_improve = 0

    # GradScaler só é relevante com AMP
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    stamp   = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(RUNS_DIR, f"{run_name}_{stamp}")
    writer  = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):
        # Freeze schedule
        if freeze_mode == "full":
            backbone_frozen = False
        elif freeze_mode == "frozen":
            backbone_frozen = True
        else:  # finetune
            backbone_frozen = epoch < unfreeze_epoch
            if epoch == unfreeze_epoch:
                print(f"[época {epoch+1}] descongelando backbone (fine-tuning completo)")

        _apply_freeze(model, backbone_frozen)

        # Treino
        model.train()
        if freeze_bn_always or backbone_frozen:
            _set_backbone_eval(model)

        train_loss = train_bce = train_rank = 0.0
        train_true_buf, train_pred_buf = [], []
        optimizer.zero_grad(set_to_none=True)  # set_to_none economiza memória

        for step, (low, high) in enumerate(
            tqdm(train_loader, desc=f"época {epoch+1}/{epochs} [treino]", leave=False)
        ):
            low_video,  _, low_mel,  low_targets  = low
            high_video, _, high_mel, high_targets = high

            # Transferência assíncrona (non_blocking=True): CPU não espera GPU
            low_video   = low_video.to(device,  non_blocking=True)
            high_video  = high_video.to(device, non_blocking=True)
            low_mel     = low_mel.to(device,    non_blocking=True)
            high_mel    = high_mel.to(device,   non_blocking=True)
            low_targets = low_targets.to(device).float().view(-1)
            high_targets= high_targets.to(device).float().view(-1)

            with torch.amp.autocast("cuda", enabled=use_amp):
                low_out  = model(low_video,  low_mel).reshape(-1)
                high_out = model(high_video, high_mel).reshape(-1)
                loss, bce_l, rank_l = criterion(
                    low_targets, high_targets, low_out, high_out,
                    return_components=True,
                )
                # Normaliza pela acumulação para gradientes comparáveis
                loss = loss / accum_steps

            scaler.scale(loss).backward()

            if (step + 1) % accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    grad_clip,
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            bs = low_video.size(0)
            # Detach + cpu imediato libera VRAM do tensor de saída
            preds   = torch.cat([torch.sigmoid(low_out), torch.sigmoid(high_out)]).detach().cpu()
            targets = torch.cat([low_targets, high_targets]).detach().cpu()

            train_loss += loss.item() * accum_steps * bs
            train_bce  += bce_l.item() * bs
            train_rank += rank_l.item() * bs

            train_true_buf.extend(targets.numpy())
            train_pred_buf.extend(preds.numpy())

        # Passo residual se o dataset não for múltiplo de accum_steps
        if (len(train_loader) % accum_steps) != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), grad_clip
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        n = len(train_loader.dataset)
        train_loss /= n; train_bce /= n; train_rank /= n

        train_true = np.array(train_true_buf)
        train_pred = np.array(train_pred_buf)
        train_mae  = np.mean(np.abs(train_true - train_pred))
        train_ccc  = calculate_ccc(train_true, train_pred)
        train_acc  = binary_accuracy(train_true, train_pred)

        # ── Validação ────────────────────────────────────────────────────────
        model.eval()
        val_loss = val_bce = val_rank = 0.0
        all_true, all_pred = [], []

        with torch.no_grad():
            for (low, high) in tqdm(val_loader, desc=f"época {epoch+1}/{epochs} [val]", leave=False):
                low_video,  _, low_mel,  low_targets  = low
                high_video, _, high_mel, high_targets = high

                low_video   = low_video.to(device,  non_blocking=True)
                high_video  = high_video.to(device, non_blocking=True)
                low_mel     = low_mel.to(device,    non_blocking=True)
                high_mel    = high_mel.to(device,   non_blocking=True)
                low_targets = low_targets.to(device).float().view(-1)
                high_targets= high_targets.to(device).float().view(-1)

                with torch.amp.autocast("cuda", enabled=use_amp):
                    low_out  = model(low_video,  low_mel).reshape(-1)
                    high_out = model(high_video, high_mel).reshape(-1)
                    loss, bce_l, rank_l = criterion(
                        low_targets, high_targets, low_out, high_out,
                        return_components=True,
                    )

                bs = low_video.size(0)
                val_loss += loss.item() * bs
                val_bce  += bce_l.item() * bs
                val_rank += rank_l.item() * bs

                preds   = torch.cat([torch.sigmoid(low_out), torch.sigmoid(high_out)]).cpu()
                targets = torch.cat([low_targets, high_targets]).cpu()
                all_true.extend(targets.numpy())
                all_pred.extend(preds.numpy())

        n = len(val_loader.dataset)
        val_loss /= n; val_bce /= n; val_rank /= n

        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        val_mae  = np.mean(np.abs(all_true - all_pred))
        val_ccc  = calculate_ccc(all_true, all_pred)
        val_acc  = binary_accuracy(all_true, all_pred)

        scheduler.step(val_loss)

        # TensorBoard
        writer.add_scalars("loss",      {"train": train_loss, "val": val_loss},  epoch)
        writer.add_scalars("bce",       {"train": train_bce,  "val": val_bce},   epoch)
        writer.add_scalars("rank_loss", {"train": train_rank, "val": val_rank},  epoch)
        writer.add_scalars("ccc",       {"train": train_ccc,  "val": val_ccc},   epoch)
        writer.add_scalars("mae",       {"train": train_mae,  "val": val_mae},   epoch)
        writer.add_scalars("acc",       {"train": train_acc,  "val": val_acc},   epoch)
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)

        print(
            f"[{epoch+1:03d}] loss={val_loss:.4f} ccc={val_ccc:.4f} "
            f"mae={val_mae:.4f} acc={val_acc:.3f}"
        )

        # Checkpoint / early-stopping
        current_score = {"ccc": val_ccc, "mae": val_mae, "loss": val_loss}[monitor]
        torch.save(model.state_dict(), last_path)

        if _is_better(current_score, best_score, mode):
            best_score   = current_score
            best_metrics = {
                "val_ccc": val_ccc, "val_mae": val_mae,
                "val_loss": val_loss, "val_acc": val_acc,
                "val_bce": val_bce, "val_rank": val_rank,
            }
            best_true, best_pred = all_true, all_pred
            torch.save(model.state_dict(), checkpoint_path)
            epochs_no_improve = 0
            print(f"  ✓ novo melhor {monitor}: {best_score:.4f} → salvo em {checkpoint_path}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping na época {epoch+1}")
                break

        # Libera tensores intermediários que possam ter ficado no grafo
        gc.collect()

    if best_true is not None:
        _log_pred_scatter(writer, best_true, best_pred, "best_pred_scatter", 0)

    writer.add_hparams(
        {
            "backbone": "r2plus1d_18", "lr": lr,
            "loss": loss_name, "monitor": monitor,
        },
        {
            "best/val_loss": best_metrics.get("val_loss", np.inf),
            "best/val_ccc":  best_metrics.get("val_ccc",  0.0),
            "best/val_acc":  best_metrics.get("val_acc",  0.0),
        },
        run_name=".",
    )
    writer.close()
    torch.cuda.empty_cache()
    print(f"Concluído. Melhor {monitor}: {best_score:.4f}")
    return {"checkpoint": checkpoint_path, "best_metrics": best_metrics}


## 8. AudioMAE — carrega e congela

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_ae = AutoModel.from_pretrained("hance-ai/audiomae", trust_remote_code=True).to(device)

# Congela o AudioMAE permanentemente
model_ae.eval()
for p in model_ae.parameters():
    p.requires_grad = False

# Confirma memória disponível antes de treinar
if torch.cuda.is_available():
    free_mb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_allocated()) / 1024**2
    print(f"VRAM livre antes do treino: {free_mb:.0f} MB")


VRAM livre antes do treino: 19851 MB


## 9. Teste rápido (sanity check)

Roda um único batch pelo modelo para conferir shapes e detectar erros antes
de gastar tempo no treino completo: útil porque a
entrada da (2+1)D CNN é diferente da ResNet2D (clipe inteiro em vez de frame
a frame).

In [13]:
_sanity_model = R2Plus1DLSTM_MultiModal(model_ae, **MODEL_DEFAULTS).to(device)
_sanity_model.eval()

with torch.no_grad():
    (low, high) = next(iter(train_loader))
    low_video, _, low_mel, low_targets = low
    low_video = low_video.to(device)
    low_mel = low_mel.to(device)
    print("video:", low_video.shape, "mel:", low_mel.shape)
    out = _sanity_model(low_video, low_mel)
    print("saída do modelo:", out.shape)

del _sanity_model
gc.collect()
torch.cuda.empty_cache()


video: torch.Size([18, 96, 1, 224, 398]) mel: torch.Size([18, 1, 128, 256])


OutOfMemoryError: CUDA out of memory. Tried to allocate 6.46 GiB. GPU 0 has a total capacity of 19.71 GiB of which 4.60 GiB is free. Process 3228127 has 11.12 GiB memory in use. Process 228984 has 2.17 GiB memory in use. Including non-PyTorch memory, this process has 1.26 GiB memory in use. Of the allocated memory 1.03 GiB is allocated by PyTorch, and 26.12 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 10. Monta modelo, otimizador e treina

In [ ]:
model = R2Plus1DLSTM_MultiModal(
    model_ae,
    use_fusion=USE_FUSION,
    **MODEL_DEFAULTS,
).to(device)

# torch.compile: funde ops e elimina overhead Python no forward
if USE_COMPILE and hasattr(torch, "compile"):
    try:
        model = torch.compile(model, mode="reduce-overhead")
    except Exception as e:
        print(f"[AVISO] torch.compile falhou ({e}); usando modelo sem compilação.")

if LR_BACKBONE is None:
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
else:
    optimizer = AdamW(
        [
            {"params": model.cnn.parameters(),  "lr": LR_BACKBONE},
            {"params": model.lstm.parameters(), "lr": LR},
            {"params": model.head.parameters(), "lr": LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )

scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)
criterion = CombinedLoss(alpha=ALPHA, margin_scale=MARGIN_SCALE)

run_name = f"r2plus1d18__{FREEZE_MODE}__fusion{USE_FUSION}__bn{ALWAYS_BN}"
print(f"=== {run_name} ===")


In [ ]:
result = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    run_name=run_name,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_name=criterion.__class__.__name__,
    freeze_mode=FREEZE_MODE,
    unfreeze_epoch=UNFREEZE_EPOCH,
    freeze_bn_always=ALWAYS_BN,
    epochs=EPOCHS,
    patience=PATIENCE,
    use_amp=USE_AMP,
    accum_steps=GRAD_ACCUM,
    lr=LR,
)

print(result["best_metrics"])


## 11. Limpeza

In [ ]:
del model, optimizer, scheduler, criterion
gc.collect()
torch.cuda.empty_cache()
